# HP_06_02_NB_BronzeToSilver_DQ — Independent quality audit

Run **after** the loading notebook, with the *same* `PipelineRunId`, `EntityName`, and `SourceName`. This notebook does not transform or merge data into Silver and does not create entity quarantine tables. It reads Silver/Quarantine and Bronze (only for source warning DQ009), and writes `dq.Rule`, `dq.Run`, `dq.Result`, and `dq.RejectedRecord`.

**Important:** Blocking validation remains in the loader's transformation functions, so bad rows never enter Silver. DQ is a separate *audit/reporting* step, not a second source of row rejection. For exact per-run counts with MERGE, the loader should execute and DQ should audit immediately, before a subsequent run overwrites `_pipeline_run_id` on the same keys.

Bronze patient gender warning (DQ009) is evaluated only if a Bronze source with `_pipeline_run_id` is available; otherwise it is skipped rather than reported as passing.


## 1. Imports

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
from delta.tables import DeltaTable
from datetime import datetime, timezone
import uuid
import re


## Pipeline parameters

In [ ]:
# Override these three parameters from the Fabric Pipeline Notebook activity.
PipelineRunId = "test11"
EntityName = "bronze_patients"
SourceName = "SYNTHEA"


## 3. Source / Target Configuration

In [ ]:
# Silver and Bronze Lakehouses (attach both Lakehouses to this notebook).
SILVER_LAKEHOUSE = "HP_02_LH_Health_Silver"
BRONZE_LAKEHOUSE = "HP_01_LH_Health_Bronze"
# Optional: specify the exact fully qualified Bronze table when its location differs.
# Example: BronzeTable = "HP_01_LH_Health_Bronze.bronze.bronze_patients"
BronzeTable = ""


## DQ schema and audit tables

In [ ]:
# DQ audit storage is in the Silver Lakehouse, independent of the Silver clinical tables.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_LAKEHOUSE}.dq")


#### Create the four DQ tables

In [ ]:
# Create audit tables before the DQ rule catalog MERGE executes.
# Only missing tables are created; existing tables are not overwritten.
DQ_TABLE_DDL = {
    "Rule": """RuleID STRING, RuleName STRING, EntityName STRING, Description STRING,
              Severity STRING, RuleType STRING, IsBlocking BOOLEAN, IsActive BOOLEAN,
              IsApplicable BOOLEAN, Notes STRING""",
    "Run": """DQRunID STRING, PipelineRunID STRING, EntityName STRING,
             StartTime TIMESTAMP, EndTime TIMESTAMP, TotalRecords BIGINT,
             ValidRecords BIGINT, RejectedRecords BIGINT, DQScore DOUBLE, Status STRING""",
    "Result": """DQRunID STRING, PipelineRunID STRING, EntityName STRING,
                RuleID STRING, Severity STRING, RecordsChecked BIGINT,
                RecordsFailed BIGINT, FailurePct DOUBLE, ExecutionTimestamp TIMESTAMP""",
    "RejectedRecord": """DQRunID STRING, PipelineRunID STRING, EntityName STRING,
                        RecordKey STRING, RuleID STRING, Severity STRING,
                        FailureReason STRING, Record STRING, FailureTimestamp TIMESTAMP"""
}
for table_name, columns in DQ_TABLE_DDL.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER_LAKEHOUSE}.dq.{table_name} ({columns}) USING DELTA")


## Data-quality rule catalog

In [ ]:
# ============================================================
# DQ RULE CATALOG
# ============================================================
# IsBlocking=True  -> record is rejected from Silver.
# IsBlocking=False -> issue is logged but record can continue.

DQ_RULES = [
    ("DQ001", "PatientIDNotNull", "Patient",
     "PatientID must not be null",
     "CRITICAL", "COMPLETENESS", True, True, True, None),

    ("DQ002", "ValidBirthDate", "Patient",
     "BirthDate must be on or before the current date",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ003", "ValidDeathDate", "Patient",
     "DeathDate must be on or after BirthDate",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ004", "EncounterPatientExists", "Encounter",
     "Encounter PatientID must exist in Silver Patient",
     "ERROR", "REFERENTIAL_INTEGRITY", True, True, True, None),

    ("DQ005", "ValidEncounterDates", "Encounter",
     "Encounter StartTime must be on or before EndTime",
     "ERROR", "VALIDITY", True, True, False,
     "Not applicable: current Encounter source has no StartTime/EndTime."),

    ("DQ006", "NonNegativeEncounterCost", "Encounter",
     "Encounter cost must be greater than or equal to zero",
     "ERROR", "VALIDITY", True, True, False,
     "Not applicable: current Encounter source has no cost column."),

    ("DQ007", "ObservationPatientExists", "Observation",
     "Observation PatientID must exist in Silver Patient",
     "ERROR", "REFERENTIAL_INTEGRITY", True, True, True, None),

    ("DQ008", "FacilityExists", "Encounter",
     "FacilityID must exist in Silver Facility",
     "ERROR", "REFERENTIAL_INTEGRITY", True, True, False,
     "Not applicable: current Encounter source has no FacilityID."),

    ("DQ009", "ValidGender", "Patient",
     "Gender must be M/MALE/F/FEMALE in the source",
     "WARNING", "DOMAIN", False, True, True, None),

    ("DQ010", "UniqueEncounterID", "Encounter",
     "EncounterID must be unique; the latest record is retained",
     "CRITICAL", "UNIQUENESS", True, True, True, None),

    # Existing entity-level checks
    ("DQ_FACILITY", "FacilityValidation", "Facility",
     "Facility required fields and numeric ranges must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_PROVIDER", "ProviderValidation", "Provider",
     "Provider required fields and numeric ranges must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_PAYER", "PayerValidation", "Payer",
     "Payer required fields and numeric values must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_ENCOUNTER", "EncounterValidation", "Encounter",
     "Encounter required fields and date values must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_CONDITION", "ConditionValidation", "Condition",
     "Condition required fields and dates must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_PROCEDURE", "ProcedureValidation", "Procedure",
     "Procedure required fields and dates must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_MEDICATION", "MedicationValidation", "Medication",
     "Medication required fields and dates must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_OBSERVATION", "ObservationValidation", "Observation",
     "Observation required fields and dates must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_RURAL_URBAN", "RuralUrbanValidation", "RuralUrban",
     "Rural/urban geography fields must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_USDA_INCOME", "USDAIncomeValidation", "USDAIncome",
     "USDA income geography and numeric fields must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_USDA_JOBS", "USDAJobsValidation", "usda_job",
     "USDA jobs geography and numeric fields must be valid",
     "ERROR", "VALIDITY", True, True, True, None),

    ("DQ_USDA_PEOPLE", "USDAPeopleValidation", "USDAPeople",
     "USDA people geography and numeric fields must be valid",
     "ERROR", "VALIDITY", True, True, True, None)
]

DQ_RULE_COLUMNS = [
    "RuleID",
    "RuleName",
    "EntityName",
    "Description",
    "Severity",
    "RuleType",
    "IsBlocking",
    "IsActive",
    "IsApplicable",
    "Notes"
]

dq_rules_df = spark.createDataFrame(
    DQ_RULES,
    DQ_RULE_COLUMNS
)

dq_rule_table = f"{SILVER_LAKEHOUSE}.dq.Rule"

target = DeltaTable.forName(spark, dq_rule_table)

(
    target.alias("target")
    .merge(
        dq_rules_df.alias("source"),
        "target.RuleID = source.RuleID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

# Small Python lookup used when quarantine metadata is added.
RULE_SEVERITY = {
    row[0]: row[4]
    for row in DQ_RULES
}


## Audit helper functions

In [ ]:
def get_nonblocking_failures(entity, bronze_df):
    """
    Count WARNING/INFO rules that do not reject a record.

    Currently DQ009 checks the original Patient gender value.
    """
    failures = {}

    if entity == "patient" and "GENDER" in bronze_df.columns:
        bad_gender = (
            bronze_df
            .filter(
                F.col("GENDER").isNull()
                |
                ~F.upper(
                    F.trim(F.col("GENDER"))
                ).isin(
                    "M",
                    "MALE",
                    "F",
                    "FEMALE"
                )
            )
            .count()
        )

        failures["DQ009"] = bad_gender

    return failures

def write_dq_results(
    quarantine_df,
    silver_entity_name,
    pipeline_run_id,
    dq_run_id,
    total_count,
    nonblocking_failures=None
):
    """
    Write one result row for every applicable DQ rule.

    Rules with no failures are also written with RecordsFailed = 0.
    """
    nonblocking_failures = nonblocking_failures or {}

    # Count blocking failures already present in quarantine.
    failure_counts = {}

    if total_count > 0:
        rows = (
            quarantine_df
            .groupBy("RuleID", "Severity")
            .count()
            .collect()
        )

        for row in rows:
            failure_counts[row["RuleID"]] = int(row["count"])

    # Add WARNING/INFO counts.
    failure_counts.update(nonblocking_failures)

    # Find applicable rules for this entity.
    applicable_rules = [
        row for row in DQ_RULES
        if (
            (row[0] != "DQ009" or "DQ009" in nonblocking_failures)
            and str(row[2]).lower()
            == str(silver_entity_name).lower()
            and row[7] is True
            and row[8] is True
        )
    ]

    result_rows = []

    for rule in applicable_rules:
        rule_id = rule[0]
        severity = rule[4]
        failed = int(
            failure_counts.get(rule_id, 0)
        )

        failure_pct = (
            failed / total_count * 100
            if total_count > 0
            else 0.0
        )

        result_rows.append(
            (
                dq_run_id,
                pipeline_run_id,
                silver_entity_name,
                rule_id,
                severity,
                int(total_count),
                failed,
                float(failure_pct)
            )
        )

    if not result_rows:
        return

    result_columns = [
        "DQRunID",
        "PipelineRunID",
        "EntityName",
        "RuleID",
        "Severity",
        "RecordsChecked",
        "RecordsFailed",
        "FailurePct"
    ]

    result_df = (
        spark.createDataFrame(
            result_rows,
            result_columns
        )
        .withColumn(
            "ExecutionTimestamp",
            F.current_timestamp()
        )
    )

    (
        result_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            f"{SILVER_LAKEHOUSE}.dq.Result"
        )
    )

def write_dq_rejected_records(
    quarantine_df,
    silver_entity_name,
    key_column,
    pipeline_run_id,
    dq_run_id
):
    """
    Store a central audit copy of rejected records.

    Entity-specific quarantine tables still keep the typed record.
    dq.RejectedRecord keeps the same failure in one central place.
    """

    metadata_columns = {
        "RuleID",
        "Severity",
        "FailureReason",
        "RunID",
        "DQRunID",
        "FailureTimestamp"
    }

    record_columns = [
        c for c in quarantine_df.columns
        if c not in metadata_columns
    ]

    rejected_df = (
        quarantine_df
        .select(
            F.lit(dq_run_id).alias("DQRunID"),
            F.lit(pipeline_run_id).alias("PipelineRunID"),
            F.lit(silver_entity_name).alias("EntityName"),
            F.coalesce(
                F.col(key_column).cast("string"),
                F.lit("<NULL>")
            ).alias("RecordKey"),
            F.col("RuleID"),
            F.col("Severity"),
            F.col("FailureReason"),
            F.to_json(
                F.struct(
                    *[
                        F.col(c)
                        for c in record_columns
                    ]
                )
            ).alias("Record"),
            F.col("FailureTimestamp")
        )
    )

    (
        rejected_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            f"{SILVER_LAKEHOUSE}.dq.RejectedRecord"
        )
    )

def write_dq_run(
    dq_run_id,
    pipeline_run_id,
    silver_entity_name,
    start_time,
    total_count,
    valid_count,
    rejected_count,
    dq_score
):
    """
    Write one summary row for the entity DQ run.
    """

    end_time = datetime.now(
        timezone.utc
    ).replace(tzinfo=None)

    row = [
        (
            dq_run_id,
            pipeline_run_id,
            silver_entity_name,
            start_time,
            end_time,
            int(total_count),
            int(valid_count),
            int(rejected_count),
            float(dq_score),
            "SUCCESS"
        )
    ]

    columns = [
        "DQRunID",
        "PipelineRunID",
        "EntityName",
        "StartTime",
        "EndTime",
        "TotalRecords",
        "ValidRecords",
        "RejectedRecords",
        "DQScore",
        "Status"
    ]

    run_df = spark.createDataFrame(
        row,
        columns
    )

    (
        run_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            f"{SILVER_LAKEHOUSE}.dq.Run"
        )
    )

In [ ]:
ENTITY_CONFIG = {
    "patient": {
        "silver_table": "clinical.Patient",
        "key": "PatientID"
    },
    "organization": {
        "silver_table": "reference.Facility",
        "key": "FacilityID"
    },
    "provider": {
        "silver_table": "reference.Provider",
        "key": "ProviderID"
    },
    "payer": {
        "silver_table": "reference.Payer",
        "key": "PayerID"
    },
    "encounter": {
        "silver_table": "clinical.Encounter",
        "key": "EncounterID"
    },
    "condition": {
        "silver_table": "clinical.Condition",
        "key": "ConditionID"
    },
    "procedure": {
        "silver_table": "clinical.Procedure",
        "key": "ProcedureID"
    },
    "medication": {
        "silver_table": "clinical.Medication",
        "key": "MedicationID"
    },
    "observation": {
        "silver_table": "clinical.Observation",
        "key": "ObservationID"
    },
    "rural_urban_2023": {
        "silver_table": "population.RuralUrban",
        "key": "RuralUrbanID"
    },
    "usda_income": {
        "silver_table": "population.USDAIncome",
        "key": "FIPS"
    },
    "usda_job": {
        "silver_table": "population.usda_job",
        "key": "FIPS",
        "processing_mode": "full"
    },
    "usda_people": {
        "silver_table": "population.USDAPeople",
        "key": "FIPS",
        "processing_mode": "full"
    }
}

## Read data from the completed loading run

In [ ]:
# Resolve the pipeline's source name to the logical key in ENTITY_CONFIG.
# Handles bronze_patients, patients, cms_hospitals and similar metadata entity names.
entity_aliases = {
    "patients": "patient", "organizations": "organization",
    "facilities": "organization", "hospitals": "organization", "cms_hospitals": "organization",
    "providers": "provider", "payers": "payer", "encounters": "encounter",
    "conditions": "condition", "procedures": "procedure",
    "medications": "medication", "observations": "observation",
    "rural_urban": "rural_urban_2023", "usda_jobs": "usda_job",
    "usda_people": "usda_people", "usda_incomes": "usda_income"
}
entity_raw = str(EntityName or "").strip().lower()
if not entity_raw:
    raise ValueError("EntityName must be provided by the pipeline or the parameter cell")
entity_clean = re.sub(r"^bronze_", "", entity_raw)
entity = entity_aliases.get(entity_clean, entity_clean)
if entity not in ENTITY_CONFIG:
    raise ValueError(f"No Silver target configured for EntityName={EntityName!r} (resolved={entity!r})")
print(f"EntityName={EntityName!r} -> logical entity={entity!r}")

# DQ009 reads the Bronze source's original gender. Locate it conservatively;
# don't silently attribute source-wide records to a particular pipeline run.
bronze_full_table = None
if entity == "patient":
    bronze_candidates = ([BronzeTable.strip()] if BronzeTable.strip() else []) + [
        f"{BRONZE_LAKEHOUSE}.bronze.{entity_raw}",
        f"{BRONZE_LAKEHOUSE}.dbo.{entity_raw}",
        f"{BRONZE_LAKEHOUSE}.{entity_raw}",
        f"{BRONZE_LAKEHOUSE}.bronze.bronze_patients",
        f"{BRONZE_LAKEHOUSE}.dbo.bronze_patients"
    ]
    bronze_full_table = next((t for t in dict.fromkeys(bronze_candidates)
                              if spark.catalog.tableExists(t)), None)
    if bronze_full_table is None:
        print("WARNING: No Bronze patient table found; DQ009 will not be evaluated.")

import uuid
from datetime import datetime, timezone

config = ENTITY_CONFIG[entity]
silver_entity_name = config["silver_table"].split(".")[-1]
silver_table = f"{SILVER_LAKEHOUSE}.{config['silver_table']}"
quarantine_table = f"{SILVER_LAKEHOUSE}.quarantine.{silver_entity_name}"
if not spark.catalog.tableExists(silver_table) and not spark.catalog.tableExists(quarantine_table):
    raise RuntimeError("No Silver or quarantine table found; run the loading notebook first.")

if spark.catalog.tableExists(silver_table):
    silver_df = spark.table(silver_table).filter(F.col("_pipeline_run_id") == PipelineRunId)
    valid_count = silver_df.count()
else:
    valid_count = 0

if spark.catalog.tableExists(quarantine_table):
    quarantine_df = spark.table(quarantine_table).filter(F.col("RunID") == PipelineRunId)
    rejected_count = quarantine_df.count()
else:
    quarantine_df = None
    rejected_count = 0

# For DQ009 the original Bronze GENDER must be preserved to check its source domain.
nonblocking_failures = {}
if entity == "patient" and bronze_full_table:
    bronze_source_df = spark.table(bronze_full_table)
    # Filter only records from this pipeline run when the source has run metadata.
    if "_pipeline_run_id" in bronze_source_df.columns:
        bronze_source_df = bronze_source_df.filter(F.col("_pipeline_run_id") == PipelineRunId)
    else:
        print("WARNING: Bronze has no _pipeline_run_id; skipping DQ009 to avoid inaccurate run-level counts.")
        bronze_source_df = None
    if bronze_source_df is not None:
        nonblocking_failures = get_nonblocking_failures(entity, bronze_source_df)

total_count = valid_count + rejected_count
dq_run_id = str(uuid.uuid4())
dq_start_time = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"DQ audit run={dq_run_id}, pipeline={PipelineRunId}, entity={silver_entity_name}")
print(f"Silver current-run rows={valid_count}, quarantine rows={rejected_count}")


## Write audit results and score

In [ ]:
# An empty quarantine DataFrame still needs the original rejection schema for the DQ writers.
if quarantine_df is None:
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType
    quarantine_df = spark.createDataFrame([], StructType([
        StructField(config["key"], StringType(), True),
        StructField("RuleID", StringType(), True),
        StructField("Severity", StringType(), True),
        StructField("FailureReason", StringType(), True),
        StructField("FailureTimestamp", TimestampType(), True),
    ]))

# Ensure absent optional Severity (older quarantine tables) is populated.
if "Severity" not in quarantine_df.columns:
    quarantine_df = quarantine_df.withColumn("Severity", F.lit("ERROR"))

# Quarantine is keyed by PipelineRunId. Re-running DQ with the same ID will append
# duplicate audit rows, so clear the previous audit rows for this run + entity first.
for audit_table in ("Result", "RejectedRecord", "Run"):
    full_name = f"{SILVER_LAKEHOUSE}.dq.{audit_table}"
    if spark.catalog.tableExists(full_name):
        from delta.tables import DeltaTable
        DeltaTable.forName(spark, full_name).delete(
            (F.col("PipelineRunID") == PipelineRunId) &
            (F.col("EntityName") == silver_entity_name)
        )

if rejected_count:
    write_dq_rejected_records(
        quarantine_df, silver_entity_name, config["key"], PipelineRunId, dq_run_id
    )
write_dq_results(
    quarantine_df, silver_entity_name, PipelineRunId, dq_run_id,
    total_count, nonblocking_failures
)
score = (100.0 * valid_count / total_count) if total_count else 100.0
write_dq_run(
    dq_run_id, PipelineRunId, silver_entity_name, dq_start_time,
    total_count, valid_count, rejected_count, score
)
print(f"DQ completed: total={total_count:,}, accepted={valid_count:,}, rejected={rejected_count:,}, score={score:.2f}%")
